# Phân tích độ nhạy của quy tắc CSA

**Câu hỏi:** kết luận "ngưỡng CVSS 8,0 vừa vá thừa vừa bỏ sót" có còn đúng khi đọc khuyến nghị của CSA sát hơn không?

Bài báo viết: *"treat published CVEs affecting internet-exposed systems with CVSS scores above 8.0 as requiring emergency patching"*, và trong cùng mục khuyên coi lỗ hổng *đã có mã khai thác minh chứng (PoC) công khai* là có thể bị khai thác ngay từ ngày công bố. Thống kê chính (`thong_ke_nguong_cvss_8.ipynb`) chỉ dùng cách đọc đơn giản nhất. Notebook này thử năm cách đọc:

| Mã | Cách đọc | Phạm vi lỗ hổng |
|---|---|---|
| Q0 | CVSS từ 8,0 (cách đọc của thống kê chính) | Mọi lỗ hổng |
| Q1 | CVSS **trên** 8,0, tức từ 8,1 | Mọi lỗ hổng |
| Q2 | CVSS từ 8,0 | Chỉ lỗ hổng khai thác được qua mạng (vector `AV:N`) |
| Q3 | CVSS từ 8,0 **hoặc** đã có PoC trên Exploit-DB | Mọi lỗ hổng |
| Q4 | CVSS từ 8,0 **hoặc** đã có PoC trên Exploit-DB | Chỉ lỗ hổng `AV:N` |

Dữ liệu công khai không cho biết hệ thống nào phơi ra Internet. Vector `AV:N` (tấn công qua mạng) chỉ là cách **gần đúng**: lỗ hổng khai thác được từ xa nếu hệ thống bị phơi ra.

Phần 1 lặp lại thống kê mô tả 1999–2025. Phần 2 kiểm thử ngược tại ba ngày đánh giá như `kiem_chung_cong_thuc_S.ipynb`, và so với công thức S ở cùng khối lượng vá.

Để so sánh được với thống kê chính, danh mục KEV được cắt về ngày 08/10/2026, đúng phiên bản 2026.10.08 mà thống kê chính đã dùng.

## 1. Cấu hình và hàm dùng chung

In [1]:
import os, io, re, json, gzip, urllib.request
import numpy as np
import pandas as pd
from IPython.display import display

START_YEAR, END_YEAR = 2002, 2025          # tep 2002 cua NVD chua ca lo hong tu 1999 den 2002
KEV_DEN_NGAY = "2026-10-08"                # cat KEV ve phien ban thong ke chinh da dung
NGAY_DANH_GIA = ["2023-12-31", "2024-12-31", "2025-06-30"]
THEO_DOI, SO_MU = 365, 0.3
DATA_DIR, OUT_DIR = "du_lieu_nvd", "ket_qua_do_nhay"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

NVD_FEED = "https://nvd.nist.gov/feeds/json/cve/2.0/nvdcve-2.0-{y}.json.gz"
KEV_URL = "https://raw.githubusercontent.com/cisagov/kev-data/develop/known_exploited_vulnerabilities.json"
EPSS_URL = "https://epss.empiricalsecurity.com/epss_scores-{d}.csv.gz"
EXPLOITDB_URL = "https://gitlab.com/exploit-database/exploitdb/-/raw/main/files_exploits.csv"
UA = {"User-Agent": "Nhom14-course-research"}

def tai(url, path):
    """Tai tep, neu da co tren may thi dung lai."""
    if os.path.exists(path):
        return open(path, "rb").read()
    data = urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=300).read()
    open(path, "wb").write(data)
    return data

def so(x): return f"{int(round(x)):,}".replace(",", ".")
def pct(v, k=2): return f"{v:.{k}f}".replace(".", ",")
def ty_le(a, b, k=2):
    if b == 0: return ""
    a_txt = so(a) if float(a).is_integer() else pct(a, 1)
    return f"{a_txt} / {so(b)} = {pct(100 * a / b, k)}%"

def so_bat_duoc(diem, y, k):
    """So lo hong vao KEV nam trong k lo hong diem cao nhat. Lo hong bang diem o ranh gioi
    duoc tinh theo gia tri ky vong (chon ngau nhien trong nhom bang diem)."""
    k = int(k)
    if k <= 0: return 0.0
    if k >= len(diem): return float(y.sum())
    t = np.partition(diem, len(diem) - k)[len(diem) - k]
    tren, bang = diem > t, diem == t
    return float(y[tren].sum() + (k - tren.sum()) * y[bang].sum() / bang.sum())

BANG, SO_LIEU = {}, {}

## 2. Tải dữ liệu

- **KEV:** cắt về ngày 08/10/2026.
- **NVD:** với mỗi lỗ hổng lấy điểm CVSS 3.x và vector tấn công (`attackVector`), ưu tiên điểm do NVD chấm như thống kê chính.
- **Exploit-DB:** mỗi mã khai thác ghi các mã CVE liên quan và ngày công bố; với mỗi CVE lấy ngày PoC công bố sớm nhất.

In [2]:
kev_js = json.loads(tai(KEV_URL, os.path.join(DATA_DIR, "kev.json")))
kev = pd.DataFrame(kev_js["vulnerabilities"])
kev["cveID"] = kev["cveID"].str.upper(); kev["ngay_kev"] = pd.to_datetime(kev["dateAdded"])
print(f"KEV tải về: phiên bản {kev_js.get('catalogVersion')}, {so(len(kev))} lỗ hổng")
kev = kev[kev["ngay_kev"] <= KEV_DEN_NGAY]
print(f"Sau khi cắt về {KEV_DEN_NGAY}: {so(len(kev))} lỗ hổng")

def chon(ds):
    if not ds: return None
    for m in ds:
        if m.get("type") == "Primary": return m
    return ds[0]

rows = []
for y in range(START_YEAR, END_YEAR + 1):
    js = json.loads(gzip.decompress(tai(NVD_FEED.format(y=y), os.path.join(DATA_DIR, f"nvdcve-2.0-{y}.json.gz"))))
    for it in js["vulnerabilities"]:
        c = it["cve"]
        if c.get("vulnStatus") == "Rejected": continue
        met = c.get("metrics", {})
        v3 = chon(met.get("cvssMetricV31")) or chon(met.get("cvssMetricV30"))
        rows.append({"cve": c["id"].upper(), "nam": int(c["id"].split("-")[1]), "ngay_cong_bo": c["published"][:10],
                     "cvss": v3["cvssData"]["baseScore"] if v3 else np.nan,
                     "av": v3["cvssData"].get("attackVector") if v3 else None})
nvd = pd.DataFrame(rows).drop_duplicates("cve")
nvd = nvd[(nvd["nam"] <= END_YEAR) & nvd["cvss"].notna()].copy()
nvd["ngay_cong_bo"] = pd.to_datetime(nvd["ngay_cong_bo"])
nvd["ngay_kev"] = nvd["cve"].map(dict(zip(kev["cveID"], kev["ngay_kev"])))
nvd["mang"] = nvd["av"] == "NETWORK"
print(f"NVD: {so(len(nvd))} lỗ hổng có CVSS 3.x, trong đó {ty_le(int(nvd['mang'].sum()), len(nvd))} có vector AV:N")

edb = pd.read_csv(io.BytesIO(tai(EXPLOITDB_URL, os.path.join(DATA_DIR, "files_exploits.csv"))), dtype=str)
edb["ngay"] = pd.to_datetime(edb["date_published"], errors="coerce")
poc = (edb.assign(cve=edb["codes"].fillna("").str.upper().str.findall(r"CVE-\d{4}-\d{4,}"))
          .explode("cve").dropna(subset=["cve", "ngay"]).groupby("cve")["ngay"].min())
nvd["ngay_poc"] = nvd["cve"].map(poc)
print(f"Exploit-DB: {so(len(edb))} mã khai thác, gắn với {so(len(poc))} mã CVE; "
      f"{ty_le(int(nvd['ngay_poc'].notna().sum()), len(nvd))} lỗ hổng có CVSS 3.x đã có PoC")

KEV tải về: phiên bản 2026.10.08, 1.739 lỗ hổng
Sau khi cắt về 2026-10-08: 1.739 lỗ hổng


NVD: 240.409 lỗ hổng có CVSS 3.x, trong đó 174.803 / 240.409 = 72,71% có vector AV:N


Exploit-DB: 47.170 mã khai thác, gắn với 25.091 mã CVE; 5.481 / 240.409 = 2,28% lỗ hổng có CVSS 3.x đã có PoC


## 3. Phần 1. Thống kê mô tả 1999–2025 theo năm cách đọc

Giống thống kê chính: dùng trạng thái KEV và PoC hiện có, nên đây là **mô tả**, không phải dự báo. Dòng Q0 phải trùng với thống kê chính (877 lỗ hổng KEV được ưu tiên, 693 bị bỏ sót).

In [3]:
co_kev = nvd["ngay_kev"].notna(); co_poc = nvd["ngay_poc"].notna()
QUY_TAC = {
    "Q0. CVSS từ 8,0": (nvd["cvss"] >= 8.0, np.ones(len(nvd), bool)),
    "Q1. CVSS trên 8,0": (nvd["cvss"] > 8.0, np.ones(len(nvd), bool)),
    "Q2. CVSS từ 8,0, chỉ lỗ hổng AV:N": (nvd["cvss"] >= 8.0, nvd["mang"].values),
    "Q3. CVSS từ 8,0 hoặc có PoC": ((nvd["cvss"] >= 8.0) | co_poc, np.ones(len(nvd), bool)),
    "Q4. CVSS từ 8,0 hoặc có PoC, chỉ lỗ hổng AV:N": ((nvd["cvss"] >= 8.0) | co_poc, nvd["mang"].values),
}
rows = []; SO_LIEU["phan_1"] = {}
for ten, (va, pham_vi) in QUY_TAC.items():
    va, pv, k = np.asarray(va), np.asarray(pham_vi), co_kev.values
    N, K = int(pv.sum()), int((k & pv).sum()); n = int((va & pv).sum()); a = int((va & pv & k).sum())
    rows.append([ten, so(N), ty_le(n, N), ty_le(a, K), ty_le(K - a, K), ty_le(a, n), pct(n / a, 1)])
    SO_LIEU["phan_1"][ten] = dict(N=N, K=K, phai_va=n, bat=a)
b1 = pd.DataFrame(rows, columns=["Cách đọc quy tắc", "Phạm vi", "Phải vá khẩn cấp", "Bắt được lỗ hổng KEV",
                                 "Bỏ sót lỗ hổng KEV", "Tỷ lệ bị khai thác trong nhóm phải vá",
                                 "Số lỗ hổng phải vá cho 1 lỗ hổng KEV"])
BANG["B1 Mo ta 1999-2025"] = b1
display(b1)
print(f"Lỗ hổng có CVSS đúng bằng 8,0: {so((nvd['cvss'] == 8.0).sum())}, trong đó {so(((nvd['cvss'] == 8.0) & co_kev).sum())} nằm trong KEV")
print(f"Lỗ hổng KEV có vector AV:N: {ty_le(int((co_kev & nvd['mang']).sum()), int(co_kev.sum()))}")
print(f"Lỗ hổng KEV đã có PoC trên Exploit-DB: {ty_le(int((co_kev & co_poc).sum()), int(co_kev.sum()))}")

,Cách đọc quy tắc,Phạm vi,Phải vá khẩn cấp,Bắt được lỗ hổng KEV,Bỏ sót lỗ hổng KEV,Tỷ lệ bị khai thác trong nhóm phải vá,Số lỗ hổng phải vá cho 1 lỗ hổng KEV
0,"Q0. CVSS từ 8,0",240.409,"66.722 / 240.409 = 27,75%","877 / 1.570 = 55,86%","693 / 1.570 = 44,14%","877 / 66.722 = 1,31%","76,1"
1,"Q1. CVSS trên 8,0",240.409,"65.754 / 240.409 = 27,35%","868 / 1.570 = 55,29%","702 / 1.570 = 44,71%","868 / 65.754 = 1,32%","75,8"
2,"Q2. CVSS từ 8,0, chỉ lỗ hổng AV:N",174.803,"62.879 / 174.803 = 35,97%","843 / 1.143 = 73,75%","300 / 1.143 = 26,25%","843 / 62.879 = 1,34%","74,6"
3,"Q3. CVSS từ 8,0 hoặc có PoC",240.409,"69.734 / 240.409 = 29,01%","1.050 / 1.570 = 66,88%","520 / 1.570 = 33,12%","1.050 / 69.734 = 1,51%","66,4"
4,"Q4. CVSS từ 8,0 hoặc có PoC, chỉ lỗ hổng AV:N",174.803,"64.875 / 174.803 = 37,11%","906 / 1.143 = 79,27%","237 / 1.143 = 20,73%","906 / 64.875 = 1,40%","71,6"


Lỗ hổng có CVSS đúng bằng 8,0: 968, trong đó 9 nằm trong KEV
Lỗ hổng KEV có vector AV:N: 1.143 / 1.570 = 72,80%
Lỗ hổng KEV đã có PoC trên Exploit-DB: 442 / 1.570 = 28,15%


## 4. Phần 2. Kiểm thử ngược tại ba ngày đánh giá

Thiết kế giống `kiem_chung_cong_thuc_S.ipynb`: tại ngày T, xét lỗ hổng đã công bố, có CVSS 3.x và chưa có trong KEV; nhãn là vào KEV trong 365 ngày sau T. PoC chỉ tính nếu đã công bố trên Exploit-DB **trước hoặc đúng ngày T**, để không nhìn trước tương lai. EPSS lấy đúng ngày T.

Với mỗi cách đọc, so sánh ba cách chọn **ở cùng khối lượng vá** với quy tắc đó: chính quy tắc, xếp theo EPSS, và xếp theo công thức S.

In [4]:
rows = []; SO_LIEU["phan_2"] = {}
for T in NGAY_DANH_GIA:
    Tt = pd.Timestamp(T); H = Tt + pd.Timedelta(days=THEO_DOI)
    raw = gzip.decompress(tai(EPSS_URL.format(d=T), os.path.join(DATA_DIR, f"epss_{T}.csv.gz"))).decode()
    ep = pd.read_csv(io.StringIO(raw), comment="#")
    x = nvd[(nvd["ngay_cong_bo"] <= Tt) & ~(nvd["ngay_kev"] <= Tt)].copy()
    x["epss"] = x["cve"].map(dict(zip(ep["cve"].str.upper(), ep["epss"]))).fillna(0.0)
    x["S"] = x["cvss"] * np.power(x["epss"], SO_MU) * 10
    x["y"] = ((x["ngay_kev"] > Tt) & (x["ngay_kev"] <= H)).astype(int)
    poc_T = x["ngay_poc"] <= Tt
    quy_tac = {
        "Q0. CVSS từ 8,0": (x["cvss"] >= 8.0, np.ones(len(x), bool)),
        "Q1. CVSS trên 8,0": (x["cvss"] > 8.0, np.ones(len(x), bool)),
        "Q2. CVSS từ 8,0, chỉ lỗ hổng AV:N": (x["cvss"] >= 8.0, x["mang"].values),
        "Q3. CVSS từ 8,0 hoặc có PoC": ((x["cvss"] >= 8.0) | poc_T, np.ones(len(x), bool)),
        "Q4. CVSS từ 8,0 hoặc có PoC, chỉ lỗ hổng AV:N": ((x["cvss"] >= 8.0) | poc_T, x["mang"].values),
    }
    for ten, (va, pv) in quy_tac.items():
        va, pv = np.asarray(va), np.asarray(pv)
        g = x[pv]; v = va[pv]; N, P, n = len(g), int(g["y"].sum()), int(v.sum())
        a = int(g["y"].values[v].sum())
        e = so_bat_duoc(g["epss"].values, g["y"].values, n); s = so_bat_duoc(g["S"].values, g["y"].values, n)
        rows.append([T, ten, ty_le(n, N), ty_le(a, P), ty_le(e, P), ty_le(s, P),
                     f"{pct((a / P) / (n / N), 1)} lần", f"{pct((s / P) / (n / N), 1)} lần"])
        SO_LIEU["phan_2"][f"{T} {ten}"] = dict(N=N, P=P, n=n, quy_tac=a, epss=e, S=s)
    print(f"{T}: {so(len(x))} lỗ hổng chưa có trong KEV, {so(int(x['y'].sum()))} vào KEV trong 12 tháng sau, "
          f"mô hình EPSS {raw.splitlines()[0].split(',')[0].replace('#model_version:', '')}")
b2 = pd.DataFrame(rows, columns=["Ngày đánh giá", "Cách đọc quy tắc", "Khối lượng vá", "Quy tắc bắt được",
                                 "EPSS cùng khối lượng", "Công thức S cùng khối lượng",
                                 "Quy tắc: gấp ngẫu nhiên", "Công thức S: gấp ngẫu nhiên"])
BANG["B2 Kiem thu nguoc"] = b2
display(b2)

2023-12-31: 147.313 lỗ hổng chưa có trong KEV, 61 vào KEV trong 12 tháng sau, mô hình EPSS v2023.03.01


2024-12-31: 186.756 lỗ hổng chưa có trong KEV, 80 vào KEV trong 12 tháng sau, mô hình EPSS v2023.03.01


2025-06-30: 209.623 lỗ hổng chưa có trong KEV, 91 vào KEV trong 12 tháng sau, mô hình EPSS v2025.03.14


,Ngày đánh giá,Cách đọc quy tắc,Khối lượng vá,Quy tắc bắt được,EPSS cùng khối lượng,Công thức S cùng khối lượng,Quy tắc: gấp ngẫu nhiên,Công thức S: gấp ngẫu nhiên
0,2023-12-31,"Q0. CVSS từ 8,0","42.685 / 147.313 = 28,98%","38 / 61 = 62,30%","49 / 61 = 80,33%","51 / 61 = 83,61%","2,1 lần","2,9 lần"
1,2023-12-31,"Q1. CVSS trên 8,0","42.224 / 147.313 = 28,66%","37 / 61 = 60,66%","49 / 61 = 80,33%","50 / 61 = 81,97%","2,1 lần","2,9 lần"
2,2023-12-31,"Q2. CVSS từ 8,0, chỉ lỗ hổng AV:N","40.622 / 107.700 = 37,72%","34 / 52 = 65,38%","48 / 52 = 92,31%","48 / 52 = 92,31%","1,7 lần","2,4 lần"
3,2023-12-31,"Q3. CVSS từ 8,0 hoặc có PoC","45.340 / 147.313 = 30,78%","42 / 61 = 68,85%","50 / 61 = 81,97%","51 / 61 = 83,61%","2,2 lần","2,7 lần"
4,2023-12-31,"Q4. CVSS từ 8,0 hoặc có PoC, chỉ lỗ hổng AV:N","42.410 / 107.700 = 39,38%","37 / 52 = 71,15%","48 / 52 = 92,31%","48 / 52 = 92,31%","1,8 lần","2,3 lần"
5,2024-12-31,"Q0. CVSS từ 8,0","52.662 / 186.756 = 28,20%","48 / 80 = 60,00%","45 / 80 = 56,25%","48 / 80 = 60,00%","2,1 lần","2,1 lần"
6,2024-12-31,"Q1. CVSS trên 8,0","51.923 / 186.756 = 27,80%","46 / 80 = 57,50%","45 / 80 = 56,25%","48 / 80 = 60,00%","2,1 lần","2,2 lần"
7,2024-12-31,"Q2. CVSS từ 8,0, chỉ lỗ hổng AV:N","49.693 / 135.157 = 36,77%","45 / 67 = 67,16%","42 / 67 = 62,69%","46 / 67 = 68,66%","1,8 lần","1,9 lần"
8,2024-12-31,"Q3. CVSS từ 8,0 hoặc có PoC","55.340 / 186.756 = 29,63%","55 / 80 = 68,75%","45 / 80 = 56,25%","49 / 80 = 61,25%","2,3 lần","2,1 lần"
9,2024-12-31,"Q4. CVSS từ 8,0 hoặc có PoC, chỉ lỗ hổng AV:N","51.495 / 135.157 = 38,10%","51 / 67 = 76,12%","42 / 67 = 62,69%","46 / 67 = 68,66%","2,0 lần","1,8 lần"


## 5. Lưu kết quả

In [5]:
with pd.ExcelWriter(os.path.join(OUT_DIR, "bang_do_nhay.xlsx")) as w:
    for ten, b in BANG.items():
        b.to_excel(w, sheet_name=ten[:31], index=False)
for ten, b in BANG.items():
    b.to_csv(os.path.join(OUT_DIR, ten.replace(" ", "_") + ".csv"), index=False, encoding="utf-8-sig")
json.dump(SO_LIEU, open(os.path.join(OUT_DIR, "so_lieu_do_nhay.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2,
          default=lambda o: int(o) if isinstance(o, np.integer) else float(o))
print("Đã lưu:", ", ".join(sorted(os.listdir(OUT_DIR))))

Đã lưu: B1_Mo_ta_1999-2025.csv, B2_Kiem_thu_nguoc.csv, bang_do_nhay.xlsx, so_lieu_do_nhay.json
